# PCA Reconstruction EDA

Dedicated notebook for pooled GTEx ground-truth PCA recovery analyses on cache-backed strict LORO rows.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.eval_utils.results_eda import *


In [ ]:
import importlib
import src.eval_utils.results_eda as results_eda
importlib.reload(results_eda)
from src.eval_utils.results_eda import *


In [ ]:
set_academic_style()

CFG = EDAConfig(
    csv_path='data/raw/gxp_samples.csv',
    hvg_path='data/raw/ahba_100hvg.txt',
    cache_root='out/loro_subject_cache',
    gene_scope='allgenes',
    naive_cache_dirname='naive',
    dlam_cache_dirname='dlam',
    plam_cache_dirname='plam',
)
CFG
print('analysis scope:', CFG.gene_scope)


## 1) Dataset Context

This notebook treats each strict cache-eval subject-parcel row as one pooled sample and each gene as a feature.

So the PCA fit matrix has shape:
- `X_true_h \in R^{n_{samples} 	imes n_{genes}}`

Here `n_samples` is the total number of strict held-out subject-parcel rows pooled across eligible subjects, not the number of subjects and not the full atlas size.


In [ ]:
PREPOST = prepare_pre_post_harmonization(CFG)
print('eligible subjects:', len(PREPOST['subjects']))
print('genes:', len(PREPOST['genes']))
print('parcels:', PREPOST['raw_cube'].shape[1])


## 2) Pooled Sample Matrices

We first pull two aligned pooled data frames from cache:
- `truth_df_*`: strict held-out truth rows
- `pred_df_*`: aligned model prediction rows on the same `(subject, parcel_idx)` keys

Why this matters:
- PCA score comparisons are only valid if true and predicted rows are ordered identically.
- The helper aligns rows using shared sample keys before any PCA is fit.


In [ ]:
MODELS = ['naive', 'dlam', 'plam']
N_JOBS = 16
EVAL_GENE_PATH = None  # set to 'ahba_100hvg', 'gtex_100hvg', 'gtex_25deg', etc.

truth_df_naive, pred_df_naive = collect_pooled_sample_prediction_dfs_from_cache_gene_subset(
    CFG,
    model='naive',
    eval_gene_path=EVAL_GENE_PATH,
    n_jobs=N_JOBS,
)
truth_df_dlam, pred_df_dlam = collect_pooled_sample_prediction_dfs_from_cache_gene_subset(
    CFG,
    model='dlam',
    eval_gene_path=EVAL_GENE_PATH,
    n_jobs=N_JOBS,
)
truth_df_plam, pred_df_plam = collect_pooled_sample_prediction_dfs_from_cache_gene_subset(
    CFG,
    model='plam',
    eval_gene_path=EVAL_GENE_PATH,
    n_jobs=N_JOBS,
)

display(truth_df_naive.head())
display(pred_df_naive.head())
print('naive shapes:', truth_df_naive.shape, pred_df_naive.shape)
print('sample keys aligned:', (truth_df_naive['sample_key'] == pred_df_naive['sample_key']).all())


In [ ]:
meta_cols = ['subject', 'parcel_idx', 'sample_key', 'sample_idx_subject', 'model', 'eval_gene_mode', 'eval_gene_path', 'mixed_space_mode']
gene_cols = [c for c in truth_df_naive.columns if c not in meta_cols]

X_true_h = truth_df_naive[gene_cols].to_numpy(dtype=np.float64)
X_pred_naive_h = pred_df_naive[gene_cols].to_numpy(dtype=np.float64)
X_pred_dlam_h = pred_df_dlam[gene_cols].to_numpy(dtype=np.float64)
X_pred_plam_h = pred_df_plam[gene_cols].to_numpy(dtype=np.float64)

print('X_true_h:', X_true_h.shape)
print('X_pred_naive_h:', X_pred_naive_h.shape)
print('X_pred_dlam_h:', X_pred_dlam_h.shape)
print('X_pred_plam_h:', X_pred_plam_h.shape)


## 3) Standard Pooled PCA

This is the default pooled PCA analysis.

What happens mathematically:
- Fit PCA on `X_true_h`
- sklearn PCA centers each gene globally across all pooled samples
- project each model prediction matrix into that same truth PCA basis
- compare true vs predicted PC scores component-wise across the pooled sample axis

Why the naive model can look nontrivial here:
- pooled PCA still contains parcel fixed effects
- naive fill reproduces parcel template structure reasonably well
- so it can score well on parcel-driven PCs even if it misses subject-specific variation


In [ ]:
pc_df, pc_summary = compute_pooled_pca_recovery_from_cache_gene_subset(
    CFG,
    num_pcs=300,
    models=MODELS,
    eval_gene_path=EVAL_GENE_PATH,
    demean_mode='none',
    n_jobs=N_JOBS,
)

display(pc_summary)
fig, ax, pc_plot_df = plot_pooled_pca_recovery(
    pc_df,
    metric='score_pearson',
    panel_label='All genes' if EVAL_GENE_PATH is None else str(EVAL_GENE_PATH),
    show_decay_fit=False,
    x_label_stride=10,
)


## 4) Within-Parcel Demeaned Pooled PCA

This version explicitly removes parcel mean structure before PCA.

Mathematically the steps are:
1. compute the truth parcel mean vector for each parcel
2. subtract that parcel mean from both truth and prediction rows within parcel
3. PCA then applies its usual global gene-wise centering on the residual matrix

Surrounding intuition:
- the first step projects out the parcel fixed effect
- the second step recenters the residual feature space for PCA

Why this should change the naive result:
- naive fill mostly models parcel-wise template effects
- after parcel demeaning, those parcel fixed effects are removed
- what remains is subject-specific within-parcel variance, where naive is expected to collapse toward zero

Why DLAM/PLAM may remain above zero:
- they generate subject-varying predictions within parcel
- so they can retain nontrivial correlation in the residualized PCA space


In [ ]:
pc_df_dm, pc_summary_dm = compute_pooled_pca_recovery_from_cache_gene_subset(
    CFG,
    num_pcs=300,
    models=MODELS,
    eval_gene_path=EVAL_GENE_PATH,
    demean_mode='within_parcel',
    n_jobs=N_JOBS,
)

display(pc_summary_dm)
fig, ax, pc_plot_df_dm = plot_pooled_pca_recovery(
    pc_df_dm,
    metric='score_pearson',
    panel_label='All genes' if EVAL_GENE_PATH is None else str(EVAL_GENE_PATH),
    show_decay_fit=False,
    x_label_stride=10,
)


## 5) Optional Decay-Fit View

A smooth decay fit can be overlaid for each model.

Why this can help:
- raw per-component points are still the real measurements
- the fitted curve gives a compact visual summary of how quickly each model's PC recovery decays across components


In [ ]:
fig, ax, pc_plot_df_dm_fit = plot_pooled_pca_recovery(
    pc_df_dm,
    metric='score_pearson',
    panel_label='All genes' if EVAL_GENE_PATH is None else str(EVAL_GENE_PATH),
    show_decay_fit=True,
    x_label_stride=10,
)


## 6) Within-Parcel Variance Check

This is a useful companion diagnostic.

Why:
- PCA recovery asks whether predicted variation aligns with the true latent axes
- within-parcel variance asks whether the model produces subject-to-subject variation at all

A model can have substantial within-parcel variance but still fail to align that variance with the dominant true PCA directions.


In [ ]:
for name, df in {
    'naive': pred_df_naive,
    'dlam': pred_df_dlam,
    'plam': pred_df_plam,
}.items():
    vals = df.groupby('parcel_idx')[gene_cols].std().mean(axis=1)
    print(f'\n{name}')
    display(vals.describe())
